In [10]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders

from Models.FraudModel import FraudMLP
from Models.AutoEncoder import AutoEncoder

from Utils.TrainUtils import get_device, train_model

In [44]:
train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


In [54]:
DEVICE = get_device()
print(DEVICE)

autoencoder = AutoEncoder (
    input_dim=input_dim,
    latent_dim=16,              
    hidden_dims=[128, 64]
).to(DEVICE)

autoencoder.load_state_dict(torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE))

# model = FraudMLP (
#     input_dim=input_dim,
#     hidden_dims=[1024, 512, 256, 128, 64],
#     gated=True,
#     dropout=0.3,
#     encoder=autoencoder,
#     freeze_encoder=True
# ).to(DEVICE)

model = FraudMLP (
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1
).to(DEVICE)

cuda


In [55]:
y = train_loader.dataset.y
pos_weight = (y == 0).sum().item() / (y == 1).sum().item()

print("Pos Weight:", pos_weight)

Pos Weight: 27.580278281911674


In [56]:
model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=130,
    lr=5e-5,
    device=DEVICE,
    pos_weight=pos_weight*0.5,
    save_path="GatedMLP_AE16_V4",
    optimize_for="f2",
    num_thresholds=300,
)

[INFO] Saved best model

Epoch 1/130
Train Loss:      0.637899
Val Loss:        0.568570
Val @ 0.5 -> Acc: 0.9241 | Prec: 0.2583 | Rec: 0.6251 | F1: 0.3655 | F2: 0.4868
Best threshold:  0.5806
Best metrics -> Acc: 0.9468 | Prec: 0.3425 | Rec: 0.5651 | F1: 0.4265 | F2: 0.5001
Confusion @ best -> TP: 1168 | FP: 2242 | TN: 54745 | FN: 899
Epoch time:      12.58s
[INFO] Saved best model

Epoch 2/130
Train Loss:      0.578576
Val Loss:        0.546190
Val @ 0.5 -> Acc: 0.9421 | Prec: 0.3256 | Rec: 0.6120 | F1: 0.4251 | F2: 0.5204
Best threshold:  0.5294
Best metrics -> Acc: 0.9491 | Prec: 0.3616 | Rec: 0.5917 | F1: 0.4489 | F2: 0.5249
Confusion @ best -> TP: 1223 | FP: 2159 | TN: 54828 | FN: 844
Epoch time:      12.78s
[INFO] Saved best model

Epoch 3/130
Train Loss:      0.543520
Val Loss:        0.537827
Val @ 0.5 -> Acc: 0.9606 | Prec: 0.4515 | Rec: 0.5830 | F1: 0.5089 | F2: 0.5509
Best threshold:  0.4620
Best metrics -> Acc: 0.9560 | Prec: 0.4130 | Rec: 0.6072 | F1: 0.4916 | F2: 0.5550


KeyboardInterrupt: 

In [50]:
# TESTING
import json

from Utils.TrainUtils import evaluate

DEVICE = get_device()

In [62]:
with open("checkpoints/GatedMLP_V5/summary.json", "r", encoding="utf-8") as f:
    summary_v2 = json.load(f)

threshold_v2 = summary_v2["best_val_metrics"]["best_threshold"]

model_v2 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=None,
    freeze_encoder=False,
).to(DEVICE)

ckpt_v2 = torch.load("checkpoints/GatedMLP_V5/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_v2:
    model_v2.load_state_dict(ckpt_v2["model_state_dict"])
else:
    model_v2.load_state_dict(ckpt_v2)

model_v2.eval()

criterion_v2 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_v2["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_v2 = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=threshold_v2,
    sweep_thresholds=False,
)

print("=== GatedMLP_V2 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2[k]}")

test_metrics_v2_fixed = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_V2 Test (0.5 Threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2_fixed[k]}")

=== GatedMLP_V2 Test ===
loss: 0.8325227823076181
threshold: 0.7083020210266113
accuracy: 0.9737867036947584
precision: 0.6132983377051037
recall: 0.6786060019328238
f1: 0.6443014655980517
f2: 0.6644549741785288
tp: 1402
fp: 884
tn: 56104
fn: 664
=== GatedMLP_V2 Test (0.5 Threshold) ===
loss: 0.8325227823076181
threshold: 0.5
accuracy: 0.964320791140149
precision: 0.4930626057512925
recall: 0.7052274927361799
f1: 0.5803624727485378
f2: 0.6493448589677415
tp: 1457
fp: 1498
tn: 55490
fn: 609


In [60]:
with open("checkpoints/GatedMLP_AE16_V4/summary.json", "r", encoding="utf-8") as f:
    summary_ae16 = json.load(f)

threshold_ae16 = summary_ae16["best_val_metrics"]["best_threshold"]

autoencoder = AutoEncoder(
    input_dim=input_dim,
    latent_dim=16,
    hidden_dims=[128, 64],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt = torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE)
if "model_state_dict" in ae_ckpt:
    autoencoder.load_state_dict(ae_ckpt["model_state_dict"])
else:
    autoencoder.load_state_dict(ae_ckpt)
autoencoder.eval()

model_ae16 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=autoencoder,
    freeze_encoder=True,
).to(DEVICE)

ckpt_ae16 = torch.load("checkpoints/GatedMLP_AE16_V4/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_ae16:
    model_ae16.load_state_dict(ckpt_ae16["model_state_dict"])
else:
    model_ae16.load_state_dict(ckpt_ae16)

model_ae16.eval()

criterion_ae16 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_ae16["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_ae16 = evaluate(
    model=model_ae16,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=threshold_ae16,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16[k]}")

test_metrics_ae16_fixed = evaluate(
    model=model_ae16,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16 Test (0.5 threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16_fixed[k]}")

=== GatedMLP_AE16 Test ===
loss: 0.6393033988064244
threshold: 0.5784
accuracy: 0.9744301825446245
precision: 0.6213973799099503
recall: 0.6887705711486507
f1: 0.6533516938164663
f2: 0.6741519781648974
tp: 1423
fp: 867
tn: 56121
fn: 643
=== GatedMLP_AE16 Test (0.5 threshold) ===
loss: 0.6393033988064244
threshold: 0.5
accuracy: 0.970112100788944
precision: 0.557377049178203
recall: 0.7076476282637578
f1: 0.6235871138565484
f2: 0.6714430031301926
tp: 1462
fp: 1161
tn: 55827
fn: 604


In [64]:
print("\n=== Comparison on held-out test ===")
print(f"Baseline V2 F2 : {test_metrics_v2['f2']:.6f}")
print(f"AE16 F2        : {test_metrics_ae16['f2']:.6f}")
print(f"Delta          : {test_metrics_ae16['f2'] - test_metrics_v2['f2']:.6f}")

print(f"\nBaseline V2 Precision/Recall: {test_metrics_v2['precision']:.4f} / {test_metrics_v2['recall']:.4f}")
print(f"AE16 Precision/Recall       : {test_metrics_ae16['precision']:.4f} / {test_metrics_ae16['recall']:.4f}")


=== Comparison on held-out test ===
Baseline V2 F2 : 0.664455
AE16 F2        : 0.674152
Delta          : 0.009697

Baseline V2 Precision/Recall: 0.6133 / 0.6786
AE16 Precision/Recall       : 0.6214 / 0.6888
